In [109]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/sujin/PycharmProjects/gorilla/berkeley-function-call-leaderboard")
import os
import json

from openai import OpenAI
from bfcl_eval.constants.model_config import ModelConfig
from bfcl_eval.model_handler.local_inference.salesforce_llama import SalesforceLlamaHandler
from bfcl_eval.model_handler.api_inference.deepseek import DeepSeekAPIHandler
from transformers import AutoTokenizer, AutoConfig

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
os.environ["DEEPSEEK_API_KEY"] = "sk-c268e4c6710441aabf62f8b401a35816"

model = DeepSeekAPIHandler(model_name="DeepSeek-R1-0528-FC", temperature=0.001)

In [15]:
test_entry = {"id": "simple_0", "question": [[{"role": "user", "content": "Find the area of a triangle with a base of 10 units and height of 5 units."}]], "function": [{"name": "calculate_triangle_area", "description": "Calculate the area of a triangle given its base and height.", "parameters": {"type": "dict", "properties": {"base": {"type": "integer", "description": "The base of the triangle."}, "height": {"type": "integer", "description": "The height of the triangle."}, "unit": {"type": "string", "description": "The unit of measure (defaults to 'units' if not specified)"}}, "required": ["base", "height"]}}]}


outputs = model.inference_single_turn_prompting(test_entry, include_input_log=False)

In [21]:
response, meta_data = outputs
print(response)
print(meta_data["reasoning_content"])

[calculate_triangle_area(base=10, height=5)]
We are given a function calculate_triangle_area that requires base and height as integers, and an optional unit (defaults to 'units')
 The question provides base=10 units and height=5 units. Since the unit is provided in the input but the function takes unit as an optional parameter, we can include it.
 However, note that the function expects unit as a string. The description says the unit defaults to 'units' if not specified, but here we have 'units' in the input.

 We can call the function with base=10, height=5, and unit='units'. But note: the input says "10 units" and "5 units", so the unit is the same.

 Alternatively, we could omit the unit and let it default to 'units'. However, the function does not require it, so we can choose to include it or not.

 Since the question states the base and height in units, and the function defaults to 'units', we can safely omit the unit parameter and it will default to 'units'. 
 But to be explicit,

# Local

In [110]:
config = {
        "model_name": "/sujin/Models/Salesforce/Llama-xLAM-2-8b-fc-r",
        "temperature": 0.001
}

model = SalesforceLlamaHandler(**config)
model.model_path_or_id = "Llama-xLAM-2-8b-fc-r"
model.tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True, trust_remote_code=True)

config = AutoConfig.from_pretrained(config["model_name"])

if hasattr(config, "max_position_embeddings"):
    model.max_context_length = config.max_position_embeddings
elif model.tokenizer.model_max_length is not None:
    model.max_context_length = model.tokenizer.model_max_length

model.base_url = "http://localhost:8000/v1"
model.client = OpenAI(
    api_key="EMPTY",
    base_url=model.base_url,
)

In [141]:
test_path = "/sujin/PycharmProjects/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_parallel_multiple.json"
# test_path = "/sujin/PycharmProjects/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_parallel.json"
# test_path = "/sujin/PycharmProjects/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_multiple.json"
# test_path = "/sujin/PycharmProjects/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_simple.json"
with open(test_path, "r") as r:
    for line in r:
        test_dict = json.loads(line)
        print(json.dumps(test_dict, indent=4))
        outputs = model.inference_single_turn_prompting(test_dict, include_input_log=False)
        print(outputs[0])
        break

{
    "id": "parallel_multiple_0",
    "question": [
        [
            {
                "role": "user",
                "content": "Find the sum of all the multiples of 3 and 5 between 1 and 1000. Also find the product of the first five prime numbers."
            }
        ]
    ],
    "function": [
        {
            "name": "math_toolkit.sum_of_multiples",
            "description": "Find the sum of all multiples of specified numbers within a specified range.",
            "parameters": {
                "type": "dict",
                "properties": {
                    "lower_limit": {
                        "type": "integer",
                        "description": "The start of the range (inclusive)."
                    },
                    "upper_limit": {
                        "type": "integer",
                        "description": "The end of the range (inclusive)."
                    },
                    "multiples": {
                        "type": "array

In [127]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

system_prompt = """\
You are a helpful assistant that can use tools. You are developed by Salesforce xLAM team.
You have access to a set of tools. When using tools, make calls in a single JSON array: 

[{"name": "tool_call_name", "arguments": {"arg1": "value1", "arg2": "value2"}}, ... (additional parallel tool calls as needed)]

If no tool is suitable, state that explicitly. If the user's input lacks required parameters, ask for clarification. Do not interpret or respond until tool results are returned. Once they are available, process them or make additional calls if needed. For tasks that don't require tools, such as casual conversation or general advice, respond directly in plain text. The available tools are:

{
    "name": "calculate_triangle_area",
    "description": "Calculate the area of a triangle given its base and height. Note that the provided function is in Python 3 syntax.",
    "parameters": {
        "type": "dict",
        "properties": {
            "base": {
                "type": "integer",
                "description": "The base of the triangle."
            },
            "height": {
                "type": "integer",
                "description": "The height of the triangle."
            },
            "unit": {
                "type": "string",
                "description": "The unit of measure (defaults to 'units' if not specified)"
            }
        },
        "required": [
            "base",
            "height"
        ]
    }
}
"""

user_prompt = """\
Find the area of a triangle with a base of 10 units and height of 5 units.
"""

chat_response = client.chat.completions.create(
    model="Llama-xLAM-2-8b-fc-r",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=1.0,
)
print(chat_response.choices[0].message.content)

[{"name": "calculate_triangle_area", "arguments": {"base": 10, "height": 523}}]


In [135]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

system_prompt = """\
You are a helpful assistant that can generate tool functions. You are developed by Salesforce xLAM team.
Your task is to generate a function description and required parameters based on a function call example and a user request. When you generate a function description, you should follow the specific format:
<function>
{
    "name": "The name of the function",
    "description": "A brief description of what the function does.",
    "parameters": {
        "type": "data type of the parameters",
        "properties": {
            "param_1": {
                "type": "data type of param_1",
                "description": "A brief description of param_1."
            },
            "param_2": {
                "type": "data type of param_2",
                "description": "A brief description of param_2."
            },
            ...,
        },
        "required": [list all required parameters]
    }
}
</function>
"""

user_prompt = """\
Find the area of a triangle with a base of 10 units and height of 5 units.
The tool to be used is:
{"name": "calculate_triangle_area", "arguments": {"base": 10, "height": 5}}
"""

chat_response = client.chat.completions.create(
    model="Llama-xLAM-2-8b-fc-r",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.001,
)
print(chat_response.choices[0].message.content)

[{"name": "calculate_triangle_area", "arguments": {"base": 10, "height": 5}}]
